# Azure Auto ML for Image Data

This notebook is a quick tutorial/guide on how to use Azure AutoML for NLP. You can also checkout this [Microsoft tutorial](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-auto-train-nlp-models?view=azureml-api-2&tabs=python).

*Note: In order to use AutoML for NLP you need to have a **GPU compute cluster**.*

# Notebook Setup

Set project paths and load workspace MLClient.

In [3]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Found the config file in: /config.json
Overriding of current TracerProvider is not allowed


Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Attempting to instrument while already instrumented


# Prepare Data

Firt we need data. We'll work on [BBC News Dataset](https://www.kaggle.com/datasets/moazeldsokyx/bbc-news) that can be found on kaggle.

In [5]:
from pathlib import Path
# Setup paths
dataset_dir = Path(os.path.join(project_dir, "data/bbc-news"))
dataset_dir.mkdir(exist_ok=True)

## Download and extract the data locally

You can manually download the dataset from kaggle or you can use kaggle API to download it using the code snippet below. To do that you need to authenticate with Kaggle API by following instructions here: https://www.kaggle.com/docs/api.

In [6]:
import kaggle

# Authenticate kaggle api
kaggle.api.authenticate()
# Download the Intel Image Classification dataset and unzip it
kaggle.api.dataset_download_files('moazeldsokyx/bbc-news', path=dataset_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/moazeldsokyx/bbc-news


## Create Azure Data Asset

You need to prepare the data in 

In [7]:
import pandas as pd

# Paths
csv_path = dataset_dir / "bbc-text.csv"
# Load data
df = pd.read_csv(csv_path)

In [ ]:
# Split
train_df = df.sample(frac=0.8, random_state=42)
valid_df = df.drop(train_df.index)

# Prepare folders
full_dir = dataset_dir / "full"
train_dir = dataset_dir / "train"
valid_dir = dataset_dir / "valid"

full_dir.mkdir(exist_ok=True)
train_dir.mkdir(exist_ok=True)
valid_dir.mkdir(exist_ok=True)

# Save CSVs
df.to_csv(full_dir / "data.csv", index=False)
train_df.to_csv(train_dir / "data.csv", index=False)
valid_df.to_csv(valid_dir / "data.csv", index=False)

In [9]:
# Create MLTable files
for d in [full_dir, train_dir, valid_dir]:
    (d / "MLTable").write_text(
        """paths:
  - file: ./data.csv"""
    )

In [10]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes


full_data_mltable_asset = Data(
    name="bbc-news-mltable",
    path=str(full_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(full_data_mltable_asset)

train_data_mltable_asset = Data(
    name="bbc-news-train-mltable",
    path=str(train_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(train_data_mltable_asset)

valid_data_mltable_asset = Data(
    name="bbc-news-valid-mltable",
    path=str(valid_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(valid_data_mltable_asset)

full_data_mltable_asset = Data(
    name="bbc-news-mltable",
    path=str(full_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(full_data_mltable_asset)

print("Data assets registered successfully.")


Uploading full (5.06 MBs): 100%|██████████| 5057520/5057520 [00:00<00:00, 7776682.74it/s]


Uploading train (4.05 MBs): 100%|██████████| 4054277/4054277 [00:00<00:00, 24093316.78it/s]


Uploading valid (1.0 MBs): 100%|██████████| 1003284/1003284 [00:00<00:00, 2619303.12it/s]




Data assets registered successfully.


# Running Auto ML for Image Classification

## Load Inputs


Now we can load an MLTable asset we've created earlier.

In [21]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# creates a dataset based on the files in the local data folder
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:intel-image-subset-mltable:3")

# Training MLTable defined locally, with local data to be uploaded
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path=training_mltable_path)
# Validation MLTable defined locally, with local data to be uploaded
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path=validation_mltable_path)
# WITH REMOTE PATH: If available already in the cloud/workspace-blob-store
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/train")
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/valid")

## Configure classification job

In [22]:
from azure.ai.ml import automl

image_classification_job = automl.image_classification(
    compute="dmdp100-gpu-cluster",
    experiment_name="dmdp100-automl-img-classification",
    display_name="intel-imgs-subset-classification-automl",
    training_data=my_training_data_input,
    target_column_name="label"
)

In [23]:
# Set limits
image_classification_job.set_limits(
    timeout_minutes=120,
    max_trials=10,
    max_concurrent_trials=3,
)

## Run classification job

In [24]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    image_classification_job
)  

# Other

Uncategorized code snippets

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_dir),
    datastore="dmdp100",
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data = ml_client.data.create_or_update(my_data)

In [ ]:

# import json

# subset_dir = dataset_dir / "subset"
# jsonl_path = subset_dir / "train_annotations.jsonl"

# records = []

# for class_dir in subset_dir.iterdir():
#     if class_dir.is_dir():
#         label = class_dir.name
#         for img_path in class_dir.glob("*.jpg"):
#             record = {
#                 "image_url": str(img_path.resolve()),  # full path
#                 "label": label
#             }
#             records.append(record)

# # Write to JSONL
# with open(jsonl_path, "w", encoding="utf-8") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ JSONL created at: {jsonl_path}")
# print(f"Total records: {len(records)}")

In [ ]:
import json

img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
base_uri = img_data_asset.path

annotations_dir = subset_dir.parent / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)